[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/certified-journeys/certified-journeys.github.io/blob/main/courses/prefect-certified/notebooks/day-11-notifications-states.ipynb#scrollTo=a2b3c4d5)

---
# Day 11 · Notifications — Email, Slack, and PagerDuty on Flow State Changes
**certified-journeys / prefect-certified** · Learn

> **Goal for today:** Wire up Slack and email notifications triggered by flow state changes, implement in-process `on_failure` and `on_completion` hooks for immediate callbacks, and understand when to choose hooks vs automations.


In [ ]:
%pip install -q "prefect>=2.14" requests


---
## Step 1 · Two notification mechanisms — hooks vs automations

Prefect gives you two independent ways to react to flow state changes:

| Mechanism | Where it runs | Latency | Best for |
|---|---|---|---|
| **Flow run hooks** (`on_failure`, `on_completion`, `on_cancellation`, `on_crashed`) | Inside the worker process, inline with the flow | Immediate (< 1 s) | In-process callbacks: write to a DB, log to Slack, clean up temp files |
| **Automations** (Prefect Cloud / server) | Server-side, triggered after run state is recorded | 5–30 s | Cross-flow triggers, time-delayed alerts, email/Slack via managed blocks |

**Rule of thumb:**
- Use **hooks** when the callback must happen in the same process (access to secrets, local filesystem)
- Use **automations** when you want server-managed routing, retries, or cross-flow logic

You can use both simultaneously — a hook sends an immediate Slack DM to the on-call engineer while an automation logs the failure to a ticketing system 30 seconds later.


In [ ]:
# Explore Prefect's built-in state types — these are the signals that trigger both hooks and automations

from prefect.states import (
    Completed,
    Failed,
    Crashed,
    Cancelled,
    Running,
    Scheduled,
    Pending,
)
from prefect.client.schemas.objects import StateType

# Print all state types with their meaning
state_info = {
    StateType.COMPLETED:   "Flow finished without error — all tasks succeeded",
    StateType.FAILED:      "Flow raised an unhandled exception",
    StateType.CRASHED:     "Worker process died unexpectedly (OOM, SIGKILL)",
    StateType.CANCELLED:   "Run was cancelled via CLI/UI/API before completion",
    StateType.RUNNING:     "Currently executing on a worker",
    StateType.SCHEDULED:   "Queued — waiting for a worker to pick it up",
    StateType.PENDING:     "Accepted by server — not yet scheduled",
}

print("Prefect state type reference:")
print("-" * 72)
for state_type, description in state_info.items():
    terminal = state_type in (StateType.COMPLETED, StateType.FAILED, StateType.CRASHED, StateType.CANCELLED)
    print(f"  {state_type.value:<14} {'[terminal]' if terminal else '          '}  {description}")

print()
print("Hook triggers (on_ arguments to @flow):")
hooks = [
    ("on_completion",   "COMPLETED",                 "Flow finished successfully"),
    ("on_failure",      "FAILED",                    "Unhandled exception in the flow"),
    ("on_cancellation", "CANCELLED",                 "Run was cancelled"),
    ("on_crashed",      "CRASHED",                   "Worker process crashed (rare)"),
    ("on_running",      "RUNNING",                   "Flow just started execution"),
]
for hook_name, state_type, when in hooks:
    print(f"  @flow({hook_name}=[...])  → fires on state={state_type:<14} ({when})")


**What just happened?**
- Prefect has 7 state types — hooks only fire on the 4 terminal states plus `RUNNING`
- **Terminal states** (Completed, Failed, Crashed, Cancelled) are final — a run cannot transition away from them
- `CRASHED` is distinct from `FAILED`: a crash means the worker process itself died (out of memory, signal kill) — the flow had no chance to raise a Python exception
- Automations can trigger on any state transition, not just terminal ones


---
## Step 2 · Implementing `on_failure` and `on_completion` hooks

A hook is a regular Python function with the signature `(flow, flow_run, state) -> None`. Prefect calls it synchronously after the flow reaches the target state.

```python
def my_hook(flow, flow_run, state):
    # flow      → the @flow-decorated function object
    # flow_run  → FlowRun schema object (id, name, parameters, tags, ...)
    # state     → State object (type, message, timestamp, result)
    ...
```

You can pass multiple hooks as a list — they run in order:
```python
@flow(on_failure=[log_failure, page_on_call, update_dashboard])
def my_flow(): ...
```


In [ ]:
from prefect import flow, task, get_run_logger
from prefect.testing.utilities import prefect_test_harness
from prefect.client.schemas.objects import FlowRun, Flow
from prefect.states import State
import datetime

# ── Hook implementations ──────────────────────────────────────────────────────

def log_failure_details(flow: Flow, flow_run: FlowRun, state: State) -> None:
    """Log structured failure information — production hook would post to Slack."""
    error_msg = state.message or "No error message captured"
    print(f"[on_failure] Flow '{flow.name}' run '{flow_run.name}' FAILED")
    print(f"  Run ID:     {flow_run.id}")
    print(f"  Tags:       {flow_run.tags}")
    print(f"  Error:      {error_msg}")
    print(f"  Timestamp:  {datetime.datetime.utcnow().isoformat()}")
    # Production equivalent:
    # post_slack_message(channel="#alerts-data", text=f"...")
    # send_pagerduty_event(severity="error", summary=error_msg)

def log_completion_details(flow: Flow, flow_run: FlowRun, state: State) -> None:
    """Log completion details — production hook would update a dashboard or DB row."""
    print(f"[on_completion] Flow '{flow.name}' run '{flow_run.name}' COMPLETED")
    print(f"  Run ID:    {flow_run.id}")
    print(f"  Timestamp: {datetime.datetime.utcnow().isoformat()}")
    # Production equivalent:
    # update_run_status_in_db(run_id=flow_run.id, status="completed")

def cleanup_temp_files(flow: Flow, flow_run: FlowRun, state: State) -> None:
    """Clean up temp resources regardless of outcome — safe to add to both hooks."""
    print(f"[cleanup] Releasing resources for run '{flow_run.name}'")
    # Production: delete S3 temp objects, close DB connections, etc.

# ── Flow using both hooks ─────────────────────────────────────────────────────

@task
def risky_task(should_fail: bool) -> str:
    if should_fail:
        raise RuntimeError("Simulated data quality failure: null values in required column")
    return "task_output_data"

@flow(
    name="pipeline-with-hooks",
    log_prints=True,
    on_failure=[log_failure_details, cleanup_temp_files],
    on_completion=[log_completion_details, cleanup_temp_files],
)
def pipeline_with_hooks(should_fail: bool = False) -> str:
    result = risky_task(should_fail=should_fail)
    return result

# ── Run both scenarios ────────────────────────────────────────────────────────
with prefect_test_harness():
    print("=" * 56)
    print("Scenario A: successful run")
    print("=" * 56)
    pipeline_with_hooks(should_fail=False)

    print()
    print("=" * 56)
    print("Scenario B: failing run (hooks fire on_failure)")
    print("=" * 56)
    try:
        pipeline_with_hooks(should_fail=True)
    except Exception:
        pass  # Expected — we're demonstrating the failure path


**What just happened?**
- Both `on_failure` and `on_completion` accept a **list** of callables — they execute in order, synchronously
- **`cleanup_temp_files`** is shared across both hook lists — a pattern for resource cleanup that must happen regardless of outcome
- Hook arguments `(flow, flow_run, state)` are injected by Prefect — never call the hook directly from user code
- Exceptions inside a hook are logged but do not change the flow's final state


---
## Step 3 · Slack notification — Webhook block pattern

Prefect **Blocks** are encrypted, workspace-scoped configuration objects. A `SlackWebhook` block stores your Incoming Webhook URL — once registered you reference it by name, never hardcode the URL.

**Setup steps (one-time per workspace):**
1. Create an Incoming Webhook app in your Slack workspace → get the URL
2. Register the block: `SlackWebhook(url="https://hooks.slack.com/...").save("my-slack-hook")`
3. Reference by name in hooks or automations: `SlackWebhook.load("my-slack-hook")`

**Why blocks for secrets?**
- URL is encrypted at rest in Prefect's block storage
- Workers retrieve the block at runtime — the URL is never in your code or git history
- Block values can be rotated in the UI without redeploying flows


In [ ]:
# Simulate a Slack notification hook — production-ready pattern
# (We use a mock sender here because real Slack webhooks require network access)

import json
import urllib.request
from prefect import flow, task, get_run_logger
from prefect.testing.utilities import prefect_test_harness
from prefect.client.schemas.objects import FlowRun, Flow
from prefect.states import State

# ── Mock Slack sender (swap for real Block in production) ─────────────────────

class MockSlackSender:
    """Simulates prefect_slack.SlackWebhook without requiring a real webhook URL."""
    def __init__(self, webhook_url: str = "https://hooks.slack.com/mock/..."):
        self.webhook_url = webhook_url
        self.messages_sent: list[dict] = []

    def notify(self, text: str, channel: str = None):
        """Simulate posting to Slack — records the message for test inspection."""
        payload = {"text": text}
        if channel:
            payload["channel"] = channel
        self.messages_sent.append(payload)
        print(f"[Slack mock] Would POST to {self.webhook_url}:")
        print(f"  payload: {json.dumps(payload, indent=4)}")

# Global mock sender — in production, use SlackWebhook.load("my-slack-hook")
slack = MockSlackSender()

# ── on_failure hook that sends a Slack message ────────────────────────────────

def slack_on_failure(flow: Flow, flow_run: FlowRun, state: State) -> None:
    """Post a structured failure message to Slack.

    Production swap:
        from prefect_slack import SlackWebhook
        slack = SlackWebhook.load("my-slack-hook")
        slack.notify(text=message)
    """
    run_url = f"https://app.prefect.cloud/flow-runs/{flow_run.id}"
    error   = state.message or "No error message"
    message = (
        f":red_circle: *Flow run failed*\n"
        f"*Flow:* `{flow.name}`\n"
        f"*Run:*  `{flow_run.name}`\n"
        f"*Error:* {error[:280]}\n"
        f"<{run_url}|View run in Prefect Cloud>"
    )
    slack.notify(text=message)

# ── on_completion hook for successful runs ────────────────────────────────────

def slack_on_completion(flow: Flow, flow_run: FlowRun, state: State) -> None:
    """Post a completion summary to Slack — useful for long-running jobs."""
    run_url = f"https://app.prefect.cloud/flow-runs/{flow_run.id}"
    message = (
        f":white_check_mark: *Flow run completed*\n"
        f"*Flow:* `{flow.name}`\n"
        f"*Run:*  `{flow_run.name}`\n"
        f"<{run_url}|View run in Prefect Cloud>"
    )
    slack.notify(text=message)

# ── Flow wired up with both Slack hooks ───────────────────────────────────────

@task
def process_batch(batch_id: int, fail: bool = False) -> int:
    logger = get_run_logger()
    logger.info(f"Processing batch {batch_id}")
    if fail:
        raise ValueError(f"Batch {batch_id}: upstream data missing from source table")
    return batch_id * 100

@flow(
    name="slack-notified-pipeline",
    log_prints=True,
    on_failure=[slack_on_failure],
    on_completion=[slack_on_completion],
)
def slack_notified_pipeline(batch_id: int = 1, fail: bool = False) -> int:
    result = process_batch(batch_id=batch_id, fail=fail)
    print(f"Batch {batch_id} processed: {result} records")
    return result

with prefect_test_harness():
    print("--- Run 1: success ---")
    slack_notified_pipeline(batch_id=1, fail=False)

    print()
    print("--- Run 2: failure (Slack on_failure fires) ---")
    try:
        slack_notified_pipeline(batch_id=2, fail=True)
    except Exception:
        pass

print()
print(f"Total Slack messages sent: {len(slack.messages_sent)}")
print("\nProduction equivalent:")
print("  from prefect_slack import SlackWebhook")
print("  slack = SlackWebhook.load('my-slack-hook')")
print("  slack.notify(text=message)")


**What just happened?**
- The `MockSlackSender.notify()` captures messages in-process — identical interface to `prefect_slack.SlackWebhook`
- The run URL (`https://app.prefect.cloud/flow-runs/{flow_run.id}`) is constructed in the hook — this deep link takes on-call engineers directly to the failed run's logs
- **Swap one line** to go from mock to production: replace `MockSlackSender()` with `SlackWebhook.load("my-slack-hook")`
- `state.message` contains the exception traceback summary — truncate to 280 chars to fit Slack's message preview


---
## Step 4 · Automations — server-side event routing

**Automations** live in the Prefect Cloud / server UI. They listen for events (flow run state changes, schedule events, custom events) and fire **actions** (send notification, create flow run, pause/resume schedule, call webhook).

**Automation anatomy:**

```yaml
trigger:
  type:   "event"                     # or "metric", "reactive"
  event:  "prefect.flow-run.failed"   # event type to watch
  match:
    "prefect.resource.id": "prefect.deployment.*"  # filter: all deployments

actions:
  - type: "send-notification"
    block_document_id: "<slack-block-id>"
    body: "Flow {{ event.resource.name }} failed at {{ event.occurred }}"
```

**Hooks vs Automations decision tree:**
```
Need immediate in-process action (< 1s)?    → hook
Need to access local filesystem / secrets?  → hook
Trigger spans multiple flows/deployments?   → automation
Need time-delayed or conditional logic?     → automation
Want managed routing (Cloud does it)?       → automation
```


In [ ]:
# Model an automation configuration as a Python dict
# In Prefect Cloud this is created via UI or the REST API

import json

# Example 1: Slack alert on ANY deployment failure
slack_failure_automation = {
    "name": "Slack — any flow failed",
    "description": "Send a Slack message when any flow run enters Failed state",
    "enabled": True,
    "trigger": {
        "type": "event",
        "event": "prefect.flow-run.failed",
        "match": {
            # Empty match = applies to all flow runs in the workspace
        },
        "within": 0,          # fire immediately (0 seconds after event)
        "posture": "Reactive",  # Reactive = fire on each event; Proactive = fire if event does NOT occur
    },
    "actions": [
        {
            "type": "send-notification",
            "block_document_id": "<slack-webhook-block-id>",
            "subject": "Flow run failed",
            "body": (
                "Flow `{{ event.resource['prefect.resource.name'] }}` failed\n"
                "Run: {{ event.resource['prefect.resource.id'] }}\n"
                "Time: {{ event.occurred }}"
            ),
        }
    ],
}

# Example 2: Email on successful completion of a specific deployment
email_completion_automation = {
    "name": "Email — capstone flow completed",
    "description": "Email the team when the capstone pipeline finishes successfully",
    "enabled": True,
    "trigger": {
        "type": "event",
        "event": "prefect.flow-run.completed",
        "match": {
            # Filter to only the capstone deployment
            "prefect.resource.name": "capstone-pipeline/prod-deployment",
        },
        "within": 0,
        "posture": "Reactive",
    },
    "actions": [
        {
            "type": "send-notification",
            "block_document_id": "<email-block-id>",
            "subject": "Capstone pipeline completed ✓",
            "body": (
                "The capstone pipeline finished successfully at {{ event.occurred }}.\n"
                "Check results at https://app.prefect.cloud/flow-runs/{{ event.resource['prefect.resource.id'] }}"
            ),
        }
    ],
}

# Example 3: SLA monitoring — alert if a flow has NOT run within 25 hours
sla_automation = {
    "name": "SLA — nightly job overdue",
    "description": "Alert if the nightly pipeline hasn't completed in 25 hours",
    "enabled": True,
    "trigger": {
        "type": "event",
        "event": "prefect.flow-run.completed",
        "match": {
            "prefect.resource.name": "nightly-ingest/prod",
        },
        "within": 90000,         # 25 hours in seconds
        "posture": "Proactive",  # fires if event does NOT occur within the window
    },
    "actions": [
        {
            "type": "send-notification",
            "block_document_id": "<pagerduty-block-id>",
            "subject": "SLA breach: nightly job overdue",
            "body": "nightly-ingest has not completed in 25 hours — investigate immediately",
        }
    ],
}

automations = [
    ("Slack failure alert",          slack_failure_automation),
    ("Email completion notification", email_completion_automation),
    ("SLA monitoring",               sla_automation),
]
for label, automation in automations:
    print(f"{'=' * 56}")
    print(f"Automation: {label}")
    print(json.dumps(automation, indent=2))
    print()


**What just happened?**
- Automations use **Jinja2 template syntax** (`{{ event.resource['prefect.resource.name'] }}`) in notification bodies — Prefect interpolates live event data
- `posture: "Proactive"` inverts the trigger: fire if the event does **not** happen within `within` seconds — this is how SLA monitoring works
- `block_document_id` references an encrypted Block — you never put the Slack URL or email credentials in the automation config
- Automations are workspace-scoped — one automation in `prod` workspace does not affect `dev` workspace


---
## Step 5 · Custom `on_failure` hook — log run URL and full error

A production `on_failure` hook needs three things:
1. The **run URL** so the recipient can navigate directly to the failure
2. The **error message** (from `state.message` — the exception traceback)
3. The **flow name + run name** so it's identifiable in a busy Slack channel

Below we implement a feature-complete hook that also handles edge cases:
- What if `state.message` is `None`?
- What if the run ID hasn't been assigned yet (edge case in early failure)?
- How to include run tags for routing alerts to the right team channel?


In [ ]:
import logging
import traceback
from prefect import flow, task, get_run_logger
from prefect.testing.utilities import prefect_test_harness
from prefect.client.schemas.objects import FlowRun, Flow
from prefect.states import State

# Use stdlib logging so the hook output is visible outside Prefect's log capture
hook_logger = logging.getLogger("prefect.hooks")
logging.basicConfig(level=logging.INFO, format="%(levelname)s %(name)s: %(message)s")


def build_run_url(flow_run: FlowRun) -> str:
    """Construct the Prefect Cloud deep link for this run.

    For self-hosted: replace the base URL with your server's address.
    """
    run_id = str(flow_run.id) if flow_run.id else "unknown"
    return f"https://app.prefect.cloud/flow-runs/{run_id}"


def extract_error_summary(state: State, max_chars: int = 500) -> str:
    """Extract a readable error summary from the state object."""
    if state.message:
        return state.message[:max_chars]
    # Fallback — try to pull from state data if message is empty
    try:
        result = state.result(raise_on_failure=False)
        return str(result)[:max_chars]
    except Exception:
        return "Error details unavailable — check run logs"


def production_on_failure_hook(flow: Flow, flow_run: FlowRun, state: State) -> None:
    """Feature-complete on_failure hook — logs run URL, error, and routing tags."""
    run_url     = build_run_url(flow_run)
    error_msg   = extract_error_summary(state)
    team_tag    = next((t for t in (flow_run.tags or []) if t.startswith("team:")), "team:unknown")
    env_tag     = next((t for t in (flow_run.tags or []) if t.startswith("env:")), "env:unknown")

    # Structured log — ingested by any log aggregation tool (Datadog, CloudWatch, etc.)
    hook_logger.error(
        "Flow run FAILED | flow=%s | run=%s | team=%s | env=%s | url=%s | error=%s",
        flow.name,
        flow_run.name,
        team_tag,
        env_tag,
        run_url,
        error_msg,
    )

    # Human-readable summary for Slack / email (printed here; POST in production)
    notification_body = f"""
:rotating_light: FLOW FAILURE ALERT
Flow:    {flow.name}
Run:     {flow_run.name}
Team:    {team_tag}
Env:     {env_tag}
Error:   {error_msg}
URL:     {run_url}
""".strip()

    print("[on_failure hook] Notification body:")
    print(notification_body)

    # Route to team-specific channel based on tag
    channel_map = {
        "team:data-eng":   "#alerts-data-engineering",
        "team:ml-platform": "#alerts-ml-platform",
        "team:unknown":    "#alerts-general",
    }
    channel = channel_map.get(team_tag, "#alerts-general")
    print(f"[on_failure hook] Would post to Slack channel: {channel}")


@task
def validate_input(value: int) -> int:
    if value < 0:
        raise ValueError(f"Input validation failed: expected non-negative integer, got {value}")
    return value * 2


@flow(
    name="production-pipeline",
    log_prints=True,
    on_failure=[production_on_failure_hook],
)
def production_pipeline(value: int = 5) -> int:
    return validate_input(value=value)


with prefect_test_harness():
    print("=" * 56)
    print("Triggering a failure to see the full on_failure hook output:")
    print("=" * 56)
    try:
        production_pipeline(value=-1)
    except Exception:
        pass  # Expected — hook fires before this propagates

    print()
    print("=" * 56)
    print("Successful run (no hook fires):")
    print("=" * 56)
    result = production_pipeline(value=7)
    print(f"Result: {result}")


**What just happened?**
- `build_run_url` safely handles `None` run ID (rare but possible in very early failures)
- `extract_error_summary` has a fallback chain: `state.message` → `state.result()` → generic message
- **Tag-based routing** reads `team:` prefixed tags to route to the right Slack channel — no hardcoded channels in the hook
- The structured log line (pipe-separated key=value) is immediately parseable by Datadog, CloudWatch Insights, or any log aggregation tool


---
## Challenge

You need to build a notification system for a data pipeline that processes financial transactions.

**Requirements:**
- On failure: log the run URL, error, and the `batch_id` parameter (from `flow_run.parameters`)
- On completion: log a summary with `batch_id` and number of records processed (return value)
- The pipeline has two tasks: `validate_batch` and `process_transactions` — make `validate_batch` fail when `batch_id` is negative

**Your tasks:**
1. Implement `failure_hook(flow, flow_run, state)` that logs the run URL and extracts `batch_id` from `flow_run.parameters`
2. Implement `completion_hook(flow, flow_run, state)` that logs the batch_id and records processed
3. Write the full `financial_pipeline(batch_id: int, records: int)` flow with both hooks
4. Run it twice inside `prefect_test_harness()`: once with `batch_id=42` (success), once with `batch_id=-1` (failure)


In [ ]:
# Challenge: your solution here
from prefect import flow, task
from prefect.testing.utilities import prefect_test_harness
from prefect.client.schemas.objects import FlowRun, Flow
from prefect.states import State

def failure_hook(flow: Flow, flow_run: FlowRun, state: State) -> None:
    # YOUR CODE HERE
    pass

def completion_hook(flow: Flow, flow_run: FlowRun, state: State) -> None:
    # YOUR CODE HERE
    pass

@task
def validate_batch(batch_id: int) -> bool:
    # YOUR CODE HERE — raise ValueError when batch_id < 0
    pass

@task
def process_transactions(records: int) -> int:
    # YOUR CODE HERE — return records processed
    pass

# @flow(on_failure=[failure_hook], on_completion=[completion_hook])
# def financial_pipeline(batch_id: int, records: int) -> int:
#     YOUR CODE HERE

# Run both scenarios
# YOUR CODE HERE


---
## Day 11 key concepts recap

| Concept | What to remember |
|---|---|
| Hooks vs automations | Hooks = in-process, immediate; Automations = server-side, cross-flow |
| Hook signature | `(flow: Flow, flow_run: FlowRun, state: State) -> None` |
| Multiple hooks | Pass a list — they execute in order, synchronously |
| State types | Completed, Failed, Crashed, Cancelled — hooks fire on terminal states |
| `state.message` | Contains the exception traceback summary for Failed runs |
| `flow_run.parameters` | Access original run parameters inside the hook |
| Slack block | `SlackWebhook.load("name")` — URL encrypted in Block storage |
| Automation posture | Reactive = fire on event; Proactive = fire when event does NOT occur |
| SLA monitoring | Proactive automation with `within` window — fire if run missed its window |

> **Tip:** Flow run hooks (`on_failure`, `on_completion`) are simpler than automations for in-process callbacks — use hooks for immediate reactions and automations for cross-flow or time-delayed triggers.

---
## What's next
**Day 12** → Docker and Kubernetes: package flow dependencies in a container image, create a Docker work pool, and understand how Kubernetes work pools create per-run Job resources.

Mark Day 11 complete in your [tracker](../index.html).
